# 06 — Vector Store Lab

ChromaDB CRUD, filtros de metadados e persistência.

In [ ]:
import sys; sys.path.insert(0, '..')
from modules.embeddings.local_embedder import LocalEmbedder
from modules.vector_store.chroma_store import ChromaVectorStore
emb = LocalEmbedder()
store = ChromaVectorStore('lab_demo', persist_dir='../modules/vector_store/chroma_db')
store.reset()
print(f'Store pronto: {store.count()} docs')

## 1. CRUD básico

In [ ]:
docs = [
    ('RAG combina busca com geração', {'source': 'paper', 'year': 2020, 'topic': 'RAG'}),
    ('ChromaDB é um vector store open-source', {'source': 'docs', 'year': 2023, 'topic': 'vectordb'}),
    ('Embeddings representam semântica', {'source': 'tutorial', 'year': 2022, 'topic': 'embeddings'}),
    ('Transformers usam self-attention', {'source': 'paper', 'year': 2017, 'topic': 'LLM'}),
]
texts = [d[0] for d in docs]
metas = [d[1] for d in docs]
vecs = emb.embed_batch(texts)
store.add(texts, vecs, metas)
print(f'Adicionados {len(docs)} docs. Total: {store.count()}')

## 2. Busca Semântica

In [ ]:
q = 'como armazenar vetores de embedding?'
q_vec = emb.embed_one(q)
results = store.query(q_vec, n_results=3)
print(f'Query: "{q}"\n')
for r in results:
    print(f'  [{r["score"]:.3f}] {r["text"]} — {r["metadata"]}')

## 3. Filtros de Metadados

In [ ]:
# Filtrar por source
results = store.query(q_vec, n_results=3, where={'source': 'paper'})
print('source=paper:')
for r in results:
    print(f'  {r["text"][:50]}')

In [ ]:
# Filtrar por year >= 2022
results = store.query(q_vec, n_results=3, where={'year': {'$gte': 2022}})
print('year >= 2022:')
for r in results:
    print(f'  {r["metadata"]["year"]} | {r["text"][:50]}')

## Exercícios

1. Implemente update: delete + re-insert com metadados atualizados
2. Meça o tempo de busca com 10, 100 e 1000 documentos
3. Explore o $in operator: filtrar por múltiplos topics